# 06 — Error Analysis

This notebook supports Jianong's README §4.4 and slide 11. It assumes the baseline §3 model artifacts are available and that prediction CSVs have been exported with `python -m app.export_predictions --all --split test`.

- `results/abstractive_summaries.csv`
- `results/predictions/nb_*_test.csv`
- `results/predictions/lr_*_test.csv`
- `results/predictions/lstm_*_test.csv`
- `results/predictions/bert_*_test.csv`
- `data/multilexsum_clean.parquet`

Goal: identify five concrete cases for model disagreement, hallucination review, confidence analysis, and rare-class behavior.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RESULTS = ROOT / 'results'
PREDS = RESULTS / 'predictions'
DATA = ROOT / 'data' / 'multilexsum_clean.parquet'

clean_df = pd.read_parquet(DATA)
summ_df = pd.read_csv(RESULTS / 'abstractive_summaries.csv')

MODEL_FRAMES = {}
for model in ['nb', 'lr', 'lstm', 'bert']:
    MODEL_FRAMES[(model, 'case')] = pd.read_csv(PREDS / f'{model}_casetype_test.csv')
    MODEL_FRAMES[(model, 'action')] = pd.read_csv(PREDS / f'{model}_classaction_test.csv')

CASE_LABELS = ['Criminal Justice', 'Civil Rights & Equality', 'Healthcare & Disability', 'Immigration & Education', 'Speech & Voting']
clean_df.shape, summ_df.shape, MODEL_FRAMES[('bert', 'case')].shape


In [ ]:
case_view = clean_df[['case_id', 'long_ref', 'short_ref', 'tiny_ref', 'class_action_sought', 'case_type_grouped']].copy()
case_view = case_view.merge(summ_df[['case_id', 'long_pred', 'short_pred', 'tiny_pred']], on='case_id', how='left')

for model in ['nb', 'lr', 'lstm', 'bert']:
    case_view = case_view.merge(
        MODEL_FRAMES[(model, 'case')].add_prefix(f'{model}_case_').rename(columns={f'{model}_case_case_id': 'case_id'}),
        on='case_id', how='left'
    )
    case_view = case_view.merge(
        MODEL_FRAMES[(model, 'action')].add_prefix(f'{model}_action_').rename(columns={f'{model}_action_case_id': 'case_id'}),
        on='case_id', how='left'
    )

case_view['case_disagreement_count'] = case_view[[f'{m}_case_y_pred' for m in ['nb', 'lr', 'lstm', 'bert']]].nunique(axis=1)
case_view['action_disagreement_count'] = case_view[[f'{m}_action_y_pred' for m in ['nb', 'lr', 'lstm', 'bert']]].nunique(axis=1)
case_view.head(2)


## Candidate selectors

The cells below surface likely cases for the five required writeups. Review the text outputs manually before freezing `results/error_cases.md`.

In [ ]:
# 1. Four-model disagreement on case type
disagreement = case_view[case_view['case_disagreement_count'] > 1].copy()
(
    disagreement[['case_id', 'case_type_grouped', 'nb_case_y_pred', 'lr_case_y_pred', 'lstm_case_y_pred', 'bert_case_y_pred', 'case_disagreement_count']]
    .sort_values(['case_disagreement_count', 'bert_case_y_proba_0'], ascending=[False, False])
    .head(15)
)


In [ ]:
# 2. High-confidence but wrong (default: BERT case type)
bert_case = MODEL_FRAMES[('bert', 'case')].copy()
bert_case['max_conf'] = bert_case[[c for c in bert_case.columns if c.startswith('y_proba_')]].max(axis=1)
high_wrong = bert_case[bert_case['y_true'] != bert_case['y_pred']].sort_values('max_conf', ascending=False)
high_wrong.head(10)


In [ ]:
# 3. Low-confidence but correct (default: BERT case type)
low_correct = bert_case[bert_case['y_true'] == bert_case['y_pred']].sort_values('max_conf', ascending=True)
low_correct.head(10)


In [ ]:
# 4. Rare-class focus: Healthcare & Disability is label index 2 under the fixed ordering.
rare_cases = case_view[case_view['case_type_grouped'] == 'Healthcare & Disability'].copy()
(
    rare_cases[['case_id', 'case_type_grouped', 'nb_case_y_pred', 'lr_case_y_pred', 'lstm_case_y_pred', 'bert_case_y_pred']]
    .head(20)
)


In [ ]:
# 5. Summarization spot-check support for hallucination review
case_view[case_view['case_id'].isin(['CJ-AL-0007', 'CJ-AL-0020'])][['case_id', 'long_ref', 'long_pred']]


## Final writeup workflow

Freeze the five README / slide / markdown examples to the cases below unless the team re-exports a different test set:

1. `IM-CA-0025` — four-model disagreement on case type
2. `CJ-AL-0020` — summary divergence / unsupported detail
3. `DR-PA-0008` — high-confidence wrong
4. `NS-DC-0123` — low-confidence correct
5. `DR-CA-0033` — rare-class recovery

Keep each commentary block to 2–4 sentences grounded in the actual text and copy the final prose into `results/error_cases.md`.
